# without spliting attack

# Mount Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


# Imports

In [1]:
import numpy as np
import os
import json
import joblib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector
from tensorflow.keras.layers import TimeDistributed, Dropout
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

c:\Users\ahmed\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(


# Paths

In [2]:
BASE_PATH = "../Data/"
DATA_PATH = os.path.join(BASE_PATH, "swat_preprocessed")
MODEL_PATH = os.path.join(BASE_PATH, "models")

os.makedirs(MODEL_PATH, exist_ok=True)

# Load Windowed Data

In [3]:
X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))

X_val = np.load(os.path.join(DATA_PATH, "X_val.npy"))
y_val = np.load(os.path.join(DATA_PATH, "y_val.npy"))

X_attack = np.load(os.path.join(DATA_PATH, "X_attack.npy"))
y_attack = np.load(os.path.join(DATA_PATH, "y_attack.npy"))

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Attack:", X_attack.shape)

Train: (429240, 60, 46)
Val: (47641, 60, 46)
Attack: (449941, 60, 46)


# Sanity Check (CRITICAL)

In [4]:
print("Train anomaly %:", y_train.mean())
print("Val anomaly %:", y_val.mean())
print("Attack anomaly %:", y_attack.mean())

Train anomaly %: 0.0
Val anomaly %: 0.0
Attack anomaly %: 0.1259454017304491


# Build LSTM Autoencoder

## Model Architecture

In [5]:
# ============================================================
# LSTM Autoencoder Model
# ============================================================

def build_lstm_autoencoder(window_size, n_features):

    inputs = Input(shape=(window_size, n_features))

    # ---------- Encoder ----------
    x = LSTM(128, return_sequences=True)(inputs)
    x = Dropout(0.2)(x)

    x = LSTM(64, return_sequences=False)(x)

    # ---------- Latent Space ----------
    latent = Dense(64, activation="relu")(x)

    # ---------- Decoder ----------
    x = RepeatVector(window_size)(latent)

    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(0.2)(x)

    x = LSTM(128, return_sequences=True)(x)

    outputs = TimeDistributed(Dense(n_features))(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model

## Train Model

In [6]:
window_size = X_train.shape[1]
n_features = X_train.shape[2]

model = build_lstm_autoencoder(window_size, n_features)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 60, 46)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 60, 128)        │        89,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 60, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 60, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 60, 46)         │         5,934 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 280,942 (1.07 MB)

 Trainable params: 280,942 (1.07 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    X_train,
    epochs=50,
    batch_size=128,
    validation_data=(X_val, X_val),
    callbacks=[early_stop],
    shuffle=True
)

Epoch 1/50
3354/3354 ━━━━━━━━━━━━━━━━━━━━ 1112s 325ms/step - loss: 0.0963 - val_loss: 0.0982
Epoch 2/50
3354/3354 ━━━━━━━━━━━━━━━━━━━━ 1061s 316ms/step - loss: 0.0600 - val_loss: 0.1104
Epoch 3/50
3354/3354 ━━━━━━━━━━━━━━━━━━━━ 1046s 312ms/step - loss: 0.0540 - val_loss: 0.0982
Epoch 4/50
3354/3354 ━━━━━━━━━━━━━━━━━━━━ 1040s 310ms/step - loss: 0.0504 - val_loss: 0.0892
Epoch 5/50
3354/3354 ━━━━━━━━━━━━━━━━━━━━ 1067s 318ms/step - loss: 0.0477 - val_loss: 0.0867
Epoch 6/50
3354/3354 ━━━━━━━━━━━━━━━━━━━━ 1045s 312ms/step - loss: 0.0475 - val_loss: 0.0959
Epoch 7/50
3354/3354 ━━━━━━━━━━━━━━━━━━━━ 1050s 313ms/step - loss: 0.0459 - val_loss: 0.0884
Epoch 8/50
3354/3354 ━━━━━━━━━━━━━━━━━━━━ 1064s 317ms/step - loss: 0.0460 - val_loss: 0.1022
Epoch 9/50
  46/3354 ━━━━━━━━━━━━━━━━━━━━ 19:36 356ms/step - loss: 0.1536

## Plot Training Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(history.history['loss'], label='Training Loss', color='r')
ax.plot(history.history['val_loss'], label='Validation Loss', color='b')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MSE)')
ax.set_title('LSTM Autoencoder Training History')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Compute Reconstruction Errors

In [ ]:
import numpy as np

def reconstruction_error(model, X):

    preds = model.predict(X)

    error = np.mean(
        np.square(X - preds),
        axis=(1,2)  # across time and features
    )

    return error



In [ ]:
val_errors = reconstruction_error(model, X_val)
attack_errors = reconstruction_error(model, X_attack)

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(val_errors, bins=80, alpha=0.6, label="Normal (Validation)")
plt.hist(attack_errors[y_attack == 1], bins=80, alpha=0.6, label="Attack")

plt.legend()
plt.title("Reconstruction Error Distribution")
plt.xlabel("Reconstruction Error")
plt.ylabel("Frequency")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

y_attack_aligned = y_attack[:]

precision, recall, thresholds = precision_recall_curve(
    y_attack_aligned,
     attack_errors
)

# # Choose threshold that maximizes F1
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)

best_threshold = thresholds[best_idx]

print("Best Threshold:", best_threshold)
print("Precision:", precision[best_idx])
print("Recall:", recall[best_idx])
print("F1:", f1_scores[best_idx])

In [ ]:
threshold = np.percentile(val_errors, 99)
print("Chosen threshold:", threshold)

In [ ]:
y_pred_attack = (attack_errors > threshold).astype(int)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_attack, y_pred_attack)

print(cm)

In [ ]:
import seaborn as sns

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_attack, y_pred_attack))

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_attack, attack_errors)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_attack, attack_errors)

plt.figure(figsize=(6,6))
plt.plot(recall, precision)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_attack, y_pred_attack))
print("Precision:", precision_score(y_attack, y_pred_attack))
print("Recall:", recall_score(y_attack, y_pred_attack))
print("F1 Score:", f1_score(y_attack, y_pred_attack))